# Trabajo Práctico Nº 1
## Objetivos

- Implementar y analizar el filtro de promedio móvil (moving average) en Python.
- Aplicar el concepto de convolución en señales digitales.
- Familiarizarse con el entorno de desarrollo en Python notebooks.
- Comparar representación en punto flotante vs punto fijo, evaluando precisión y tiempos de cómputo.

# Ejercicio 1 - Filtro Moving Average

1. Implementá un algoritmo de promedio móvil (moving average) según la definición vista en clase, que reciba como parámetros:

- La señal de entrada.
- El tamaño de la ventana.

A partir de tu implementación, obtené y graficá la respuesta impulsiva del sistema (lo que en clase llamamos su “firma” o función de transferencia).

2. Respondé:

- ¿Cómo se obtiene la respuesta impulsiva de cualquier sistema?
- ¿Qué representa en el caso del filtro de promedio móvil?

In [ ]:
# Escribí tu código acá

# Ejercicio 2 - Respuesta del sistema

1. Genera dos señales de prueba:
    - Una onda cuadrada de amplitud 1 y frecuencia de 2 kHz.
    - Una onda triangular de amplitud 1 y frecuencia de 1 kHz.

Aplica el moving average implementado en el Ejercicio 1 a estas señales.

2. Analiza los resultados:

    - Grafica la señal original y la señal filtrada.
    - Explica qué cambios observas en la forma de onda.

3. Escribe tus conclusiones: ¿qué efecto tiene el filtro de promedio móvil sobre cada señal?


In [ ]:
# Escribí tu código acá

# Ejercicio 3 - Convolución en punto flotante y punto fijo

1. Implementa un algoritmo de convolución:

    - En punto flotante (float).
    - En punto fijo, utilizando por ejemplo la librería fixedpoint.

2. Compara los resultados con la función de NumPy np.convolve. Mide y compara:

    - El tiempo de ejecución usando timeit.
    - El error de la salida respecto al cálculo en punto flotante.

3. Conclusión:

    - ¿Qué diferencias observas entre las implementaciones?
    - ¿Qué ventajas y desventajas tiene usar punto fijo en lugar de punto flotante?

In [ ]:
!pip install fixedpoint

In [ ]:
from __future__ import division
import numpy as np
import math
import scipy.signal as sig
from scipy.fftpack import fft, ifft, fftshift
import matplotlib.pyplot as plt
from fixedpoint import FixedPoint

M,N = 10,3
# M = Bits enteros, N = Bits fraccionarios

a_fp = 2.61231231241
b_fp = -1.54
a_fx = FixedPoint(a_fp,n=N,m=M)
b_fx = FixedPoint(b_fp,n=N,m=M)

print (a_fx.qformat)
print (float(a_fx))
print (float(b_fx))


def fixedToFloat(array):
  result = []
  for num in array:
    result.append(float(num))
  return result

def arrayFixedPointValue(initial_value, fx_array_len, qformat):
  # Retorna: un array con initial_value en todas sus posiciones, de longitud
  # fx_array_len, con el formato qformat = {'signed': ?, 'm': ?, 'n': ?}
  fx_type = FixedPoint(initial_value, **qformat)
  fx_array = [fx_type for _ in range(fx_array_len)]
  return fx_array

def arrayFixedPoint(initial_array, qformat):
  # Retorna: un array con los valores de initial_array convertidos
  # a fixed point, de la misma longitud
  # con el formato qformat = {'signed': ?, 'm': ?, 'n': ?}
  fx_array = []
  for num in initial_array:
    fx_num = FixedPoint(num, **qformat)
    fx_array.append(fx_num)
  return fx_array

# Quiero hacer un array de fixedpoint
qformat = {'signed': True, 'm': 1, 'n': 4}
fx_array = arrayFixedPointValue(0,10,qformat)
print(fixedToFloat(fx_array))


UQ10.3
2.625
-1.5
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
from __future__ import division
import numpy as np
from fixedpoint import FixedPoint

# -----------------------------
# FUNCIONES AUXILIARES
# -----------------------------
def fixedToFloat(array):
    """Convierte un array de FixedPoint a lista de floats."""
    result = []
    for num in array:
        result.append(float(num))
    return result

def arrayFixedPointValue(initial_value, fx_array_len, qformat):
    """Crea un array de punto fijo con initial_value en todas sus posiciones."""
    fx_type = FixedPoint(initial_value, **qformat)
    fx_array = [FixedPoint(initial_value, **qformat) for _ in range(fx_array_len)]
    return fx_array

def arrayFixedPoint(initial_array, qformat):
    """Convierte un array de floats a FixedPoint con formato qformat."""
    fx_array = []
    for num in initial_array:
        fx_num = FixedPoint(num, **qformat)
        fx_array.append(fx_num)
    return fx_array

# -----------------------------
# CONVOLUCIÓN EN PUNTO FIJO
# -----------------------------
def convolucion_fixed(x, h, qformat):
    """Convolución discreta en punto fijo."""
    N = len(x)
    M = len(h)
    y_len = N + M - 1

    # Convertimos entradas a punto fijo
    x_fx = arrayFixedPoint(x, qformat)
    h_fx = arrayFixedPoint(h, qformat)

    # Inicializamos salida en punto fijo
    y_fx = arrayFixedPointValue(0, y_len, qformat)

    for n in range(y_len):
        acc = FixedPoint(0, **qformat)
        for k in range(N):
            if 0 <= n-k < M:
                acc += x_fx[k] * h_fx[n-k]
        y_fx[n] = acc
    return y_fx

# Señales de ejemplo
x = [1.0, 2.05, 3.0]
h = [1.5, -1.0, 2.0]

# Definimos formato Q -> 3 bits enteros + 4 bits fraccionarios
qformat = {'signed': True, 'm': 3, 'n': 20}

# Convolución en fixed
y_fixed = convolucion_fixed(x, h, qformat)
y0 = fixedToFloat(y_fixed)

# Resultados
#print("Resultado punto fijo (FixedPoint):", y_fixed)
print("Convertido a float:", y0)




Convertido a float: [1.5, 2.075000286102295, 4.449999809265137, 1.1000003814697266, 6.0]


In [ ]:
#1)
#0-quantization.ipynb
#punto flotante
import numpy as np

def convolucion_float(x, h):
    N = len(x)
    M = len(h)
    y = np.zeros(N + M - 1, dtype=float)

    for n in range(len(y)):
        for k in range(N):
            if 0 <= n-k < M:
                y[n] += x[k] * h[n-k]
    return y

# Ejemplo
x = [1.0, 2.05, 3.0]
h = [1.5, -1.0, 2.0]

y = convolucion_float(x, h)

print(y[0])
print(y[1])
print(y[2])
print(y[3])
print(y[4])





1.5
2.0749999999999997
4.45
1.0999999999999996
6.0


In [ ]:
#2)

In [ ]:
# Ejemplos de medicion de tiempos
# %pylab inline
import timeit
import time
import numpy as np

t = np.arange(0, 1024/20000, 1/20000)
s1= np.sin(2*np.pi*1000*t)+1
s2 = np.sin(2*np.pi*1200*t)

# METODO 1 de medición de tiempos

# Este wrapper empaqueta funciones para que el timeit pueda medir solo eso
def wrapper(func, *args):
    def wrapped():
        return func(*args)
    return wrapped

conv = wrapper(np.convolve, s1, s2)
print (timeit.timeit(conv, number=1))

# METODO 2 de medición de tiempos
start = time.time()
conv_r = np.convolve(s1,s2)
end = time.time()
print (end-start)

0.00020788999972864985
0.00029969215393066406


In [ ]:
# Ejemplos de medicion de tiempos
# %pylab inline
import timeit
import time
import numpy as np

# METODO 2 de medición de tiempos
start = time.time()
conv_r = convolucion_float(x,h)
end = time.time()
print ("punto flotante",(end-start)*100000)

start = time.time()
y_fixed = convolucion_fixed(x, h, qformat)
end = time.time()
print ("punto fijo",(end-start)*100000)

start = time.time()
conv_r = np.convolve(s1,s2)
end = time.time()
print ("numpy",(end-start)*100000)


print(y-y0)


punto flotante 8.511543273925781
punto fijo 101.99546813964844
numpy 49.09038543701172
[ 0.00000000e+00 -2.86102295e-07  1.90734863e-07 -3.81469727e-07
  0.00000000e+00]


In [ ]:
#3)Conclusión:

#¿Qué diferencias observas entre las implementaciones?
#¿Qué ventajas y desventajas tiene usar punto fijo en lugar de punto flotante?

#sistemas de numeacion



3)Conclusión:
¿Qué diferencias observas entre las implementaciones?

La principal diferencia entre punto fijo y punto flotante está en el tiempo de ejecución y la precisión. El punto fijo es más rápido de implementar, consume menos recursos y energía, lo que lo hace ideal para sistemas en tiempo real. Sin embargo, sacrifica precisión.

El punto flotante, en cambio, ofrece mucha más precisión, pero requiere más memoria, mayor potencia de cómputo y un costo de hardware más alto, además de ser más complejo de desarrollar.

